# Combined Lean Notebook — Phase 2 + 3 + 4

This is the streamlined version of the full combined notebook.

- Keep one runtime/session and run top to bottom.
- Duplicate mount/setup cells from later phases are removed.
- Phase outputs are unchanged.


---

## Section A — Phase 2


# Phase 2 — Domain Generalisation across 6 machine sections

Phase 2 trains the anomaly-detection autoencoder on the combined training pool of `dev_train` (sections 00-02) **plus** `eval_train` (sections 03-05), using **stratified K-fold cross-validation** over `(section × domain)` so every fold sees roughly the same number of clips from every bucket. The K fold-best checkpoints are averaged at inference to form an **ensemble**, which is the number we leaderboard and ship.

| split | folder | #clips | labels? | used in phase 2? |
|---|---|---|---|---|
| dev train (train)  | `data/dcase_bearing_dev/bearing/train`  | 2999 | all normal | yes — in K-fold training pool |
| eval train (train) | `data/dcase_bearing_eval/bearing/train` | 3000 | all normal | yes — in K-fold training pool |
| dev test (eval)    | `data/dcase_bearing_dev/bearing/test`   | 600  | normal/anomaly labelled | yes — ensemble evaluation |
| **eval test (eval)** | `data/dcase_bearing_eval/bearing/test`  | 600  | **labels not public** | **NO — held out until a best model is selected** |

> Eval-test is completely untouched by this notebook: not staged, not cached, not loaded. A separate submission script will score it once a best `(mode, arch)` has been picked from the phase-2 leaderboard.

### What this notebook produces

1. **Four trained K-fold ensembles** — `baseline / mixed / domain_regularized / contrastive`, each with K=5 fold checkpoints.
2. **Per-mode evaluation** on the labelled `dev_test` split with per-fold and ensemble metrics.
3. **A single ranked leaderboard + recommendation** — the collector picks the best `(mode, arch)` for you.
4. **A zip bundle** — `results_phase2.zip` — with the leaderboard, plots, every run's checkpoints, and a top-level `best_model/` folder containing the K fold-checkpoints of the recommended model. The bundle is mirrored to your Drive and also offered as a local download.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound

/content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound


In [4]:
import os, shutil, time, zipfile
import shutil


## Step 0: Stage the three splits we are allowed to use

Same approach as phase 1 — running off Drive FUSE is slow, so we stage once to `/content/` and read from there. If the zips are already extracted under `data/` we skip re-extraction.

Expected counts (verify before proceeding):
- dev train  → 2999 WAVs  (sections 00-02)
- eval train → 3000 WAVs  (sections 03-05)
- dev test   → 600  WAVs  (sections 00-02, labelled)

`eval test` (sections 03-05, unlabelled) is **not staged** — it is the held-out prediction split and must not be touched until a best DG model has been picked.

In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound

import os, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

PROJECT_ROOT = os.getcwd()
LOCAL_DATA   = "/content/bearing_data"
LOCAL_CACHE  = "/content/bearing_cache"   # dev train cache (phase 1 reused)
os.makedirs(LOCAL_DATA,  exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)

SPLITS = {
    # logical name : (project-relative src,  local dst,  source-zip,          inner prefix in zip)
    "dev_train":  ("data/dcase_bearing_dev/bearing/train",
                    f"{LOCAL_DATA}/dcase_bearing_dev/bearing/train",
                    "dev_bearing.zip",              "bearing/train/"),
    "dev_test":   ("data/dcase_bearing_dev/bearing/test",
                    f"{LOCAL_DATA}/dcase_bearing_dev/bearing/test",
                    "dev_bearing.zip",              "bearing/test/"),
    "eval_train": ("data/dcase_bearing_eval/bearing/train",
                    f"{LOCAL_DATA}/dcase_bearing_eval/bearing/train",
                    "eval_data_bearing_train.zip", "bearing/train/"),
    # NOTE: eval_test (sections 03-05, unlabelled) is intentionally NOT staged.
    # It is the final held-out prediction split and must not be touched until
    # a best DG model has been selected from the leaderboard in Step 6.
}

def _list_wavs(d):
    if not os.path.isdir(d):
        return []
    return sorted(f for f in os.listdir(d) if f.endswith(".wav"))

def _copy_parallel(src_dir, dst_dir, files):
    def _one(f):
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    with ThreadPoolExecutor(max_workers=16) as ex:
        for fut in as_completed([ex.submit(_one, f) for f in files]):
            fut.result()

def _extract_from_zip(zip_path, inner_prefix, dst_dir):
    """Extract only files under `inner_prefix` from zip_path into dst_dir (flat)."""
    print(f"  Extracting {inner_prefix} from {zip_path} ...")
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist()
                    if m.filename.startswith(inner_prefix) and m.filename.endswith('.wav')]
        t0 = time.time()
        for i, m in enumerate(members, 1):
            leaf = os.path.basename(m.filename)
            if not leaf:
                continue
            out = os.path.join(dst_dir, leaf)
            if os.path.exists(out) and os.path.getsize(out) == m.file_size:
                continue
            with zf.open(m) as fin, open(out, "wb") as fout:
                shutil.copyfileobj(fin, fout)
            if i % 500 == 0:
                print(f"    {i}/{len(members)} ({time.time()-t0:.1f}s)")
    print(f"  -> {len(_list_wavs(dst_dir))} files now in {dst_dir}")

for name, (src, dst, zip_name, inner) in SPLITS.items():
    os.makedirs(dst, exist_ok=True)
    print(f"\n== {name} ==")
    # Prefer already-extracted data folder; otherwise fall back to the zip.
    if os.path.isdir(src) and len(_list_wavs(src)) > 0:
        src_wavs = _list_wavs(src)
        dst_wavs = set(_list_wavs(dst))
        missing  = [f for f in src_wavs if f not in dst_wavs]
        print(f"  source(drive): {len(src_wavs)}  local: {len(dst_wavs)}  missing: {len(missing)}")
        if missing:
            _copy_parallel(src, dst, missing)
            print(f"  -> {len(_list_wavs(dst))} files now in {dst}")
    elif os.path.exists(zip_name):
        _extract_from_zip(zip_name, inner, dst)
    else:
        raise FileNotFoundError(
            f"Neither {src} nor {zip_name} exists for split {name}. Upload the zip"
            f" to the project root or extract into data/ first."
        )

# ---- sanity counts ----
print("\n=== staged counts ===")
for name, (_, dst, _, _) in SPLITS.items():
    print(f"  {name:<10}: {len(_list_wavs(dst))} WAVs at {dst}")

/content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound

== dev_train ==
  source(drive): 2999  local: 0  missing: 2999
  -> 2999 files now in /content/bearing_data/dcase_bearing_dev/bearing/train

== dev_test ==
  source(drive): 600  local: 0  missing: 600
  -> 600 files now in /content/bearing_data/dcase_bearing_dev/bearing/test

== eval_train ==
  source(drive): 3000  local: 0  missing: 3000
  -> 3000 files now in /content/bearing_data/dcase_bearing_eval/bearing/train

=== staged counts ===
  dev_train : 2999 WAVs at /content/bearing_data/dcase_bearing_dev/bearing/train
  dev_test  : 600 WAVs at /content/bearing_data/dcase_bearing_dev/bearing/test
  eval_train: 3000 WAVs at /content/bearing_data/dcase_bearing_eval/bearing/train


In [3]:
# ---- point the phase-2 scripts at the local-disk copies ----
os.environ["BEARING_DEV_TRAIN_DIR"]    = SPLITS["dev_train"][1]
os.environ["BEARING_DEV_TEST_DIR"]     = SPLITS["dev_test"][1]
os.environ["BEARING_EVAL_TRAIN_DIR"]   = SPLITS["eval_train"][1]

# Separate cache directories so each split has its own .npy tree.
os.environ["BEARING_DEV_TRAIN_CACHE"]  = f"{LOCAL_CACHE}_dev_train"
os.environ["BEARING_DEV_TEST_CACHE"]   = f"{LOCAL_CACHE}_dev_test"
os.environ["BEARING_EVAL_TRAIN_CACHE"] = f"{LOCAL_CACHE}_eval_train"

# Phase-2-only checkpoints (phase-1 stays untouched).
os.environ["BEARING_PHASE2_CKPT_DIR"]  = "/content/checkpoints_phase2"

# Guard: explicitly unset any eval-test env vars a previous run may have left
# behind, so no phase-2 script can accidentally read the held-out split.
for k in ("BEARING_EVAL_TEST_DIR", "BEARING_EVAL_TEST_CACHE"):
    os.environ.pop(k, None)

print("Phase-2 environment variables set:")
for k, v in sorted(os.environ.items()):
    if k.startswith("BEARING_"):
        print(f"  {k} = {v}")

Phase-2 environment variables set:
  BEARING_DEV_TEST_CACHE = /content/bearing_cache_dev_test
  BEARING_DEV_TEST_DIR = /content/bearing_data/dcase_bearing_dev/bearing/test
  BEARING_DEV_TRAIN_CACHE = /content/bearing_cache_dev_train
  BEARING_DEV_TRAIN_DIR = /content/bearing_data/dcase_bearing_dev/bearing/train
  BEARING_EVAL_TRAIN_CACHE = /content/bearing_cache_eval_train
  BEARING_EVAL_TRAIN_DIR = /content/bearing_data/dcase_bearing_eval/bearing/train
  BEARING_PHASE2_CKPT_DIR = /content/checkpoints_phase2


In [4]:
!pip install -q -r requirements.txt

## Step 1: Pre-compute mel caches for the three splits we use

Decodes every WAV exactly once to a `.npy` on local disk. Atomic writes + size sanity checks prevent the truncated-cache bug we hit in phase 1.

Expected output (first run): ~6,600 new `.npy` files across three cache dirs:
- `dev_train`  ≈ 2999 files
- `eval_train` ≈ 3000 files
- `dev_test`   ≈ 600 files

Subsequent runs say "Cache already complete" and cost nothing. The eval-test cache is **not** built — that split stays on disk as raw WAVs (or on the zip) and we do not touch it.

In [5]:
!python prepare_cache_lean.py


=== dev train: /content/bearing_data/dcase_bearing_dev/bearing/train ===
Total samples loaded: 2999 from /content/bearing_data/dcase_bearing_dev/bearing/train
Building mel cache for 2999 files -> /content/bearing_cache_dev_train
  cached 200/2999
  cached 400/2999
  cached 600/2999
  cached 800/2999
  cached 1000/2999
  cached 1200/2999
  cached 1400/2999
  cached 1600/2999
  cached 1800/2999
  cached 2000/2999
  cached 2200/2999
  cached 2400/2999
  cached 2600/2999
  cached 2800/2999
  cached 2999/2999

=== eval train: /content/bearing_data/dcase_bearing_eval/bearing/train ===
Total samples loaded: 3000 from /content/bearing_data/dcase_bearing_eval/bearing/train
Building mel cache for 3000 files -> /content/bearing_cache_eval_train
  cached 200/3000
  cached 400/3000
  cached 600/3000
  cached 800/3000
  cached 1000/3000
  cached 1200/3000
  cached 1400/3000
  cached 1600/3000
  cached 1800/3000
  cached 2000/3000
  cached 2200/3000
  cached 2400/3000
  cached 2600/3000
  cached 280

## Step 2: Train all four modes with stratified K-fold CV

`train_lean.py` loads `dev_train + eval_train` as a single `CombinedBearingDataset` and builds **stratified K=5-fold splits over (section × domain)** for both source and target clips separately. Each fold's val set is `(source_fold[k]) ∪ (target_fold[k])` — both domains, all normal — so the best-epoch signal reflects target reconstruction too.

For each fold the script trains `--epochs` epochs, saves the best-val checkpoint as `checkpoints_phase2/{mode}_{arch}_fold{k}.pt`, and writes a per-fold train/val curve PNG. A per-run `{mode}_{arch}_kfold_summary.json` + grid PNG is emitted at the end.

Defaults: 20 epochs × 5 folds, batch 64, 80/20 source/target sampler, `--seed 42` for a deterministic stratification. Expect ~5× phase-1 training time per `(mode, arch)`.

Tip for fast iteration: pass `--folds_to_run 0` to run a single fold only.

In [12]:
!python train_lean.py --mode mixed       --arch ae --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [13]:
!python train_lean.py --mode domain_regularized --arch ae --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [14]:
!python train_lean.py --mode contrastive --arch ae --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


## Step 2b: Train every mode with the **UNet** backbone

The tiny `SimpleAE` hits a ceiling around target AUC ~0.55 on dev-test (see phase-2 run 1). `UNetAE` is a 2-level U-Net with skip connections and a 128-d bottleneck (vs. 32-d for SimpleAE) — more capacity, enough to reconstruct finer spectrogram structure, still small enough to train per fold in a few minutes. Trained for **40 epochs** (up from 20 on AE) because the larger model converges a bit slower.

The four cells below run the same four DG modes — `baseline / mixed / domain_regularized / contrastive` — with `--arch unet`. Each emits `checkpoints_phase2/{mode}_unet_fold{0..4}.pt` that Step 4 evaluates automatically.

In [6]:
!python train_lean.py --mode mixed --arch unet_mobilenet_encoder --epochs 15 --n_folds 3

=== Phase-2 K-fold training ===
  train root: /content/bearing_data/dcase_bearing_dev/bearing/train  [ok]
  train root: /content/bearing_data/dcase_bearing_eval/bearing/train  [ok]

Stratified K-fold config: K=3  total_source=5940  total_target=59
  strata = (section, domain)  [12 buckets: 6 sections × 2 domains]
  source fold sizes: [1980, 1980, 1980]
  target fold sizes: [19, 20, 20]
  val set per fold  = (source fold) ∪ (target fold), both domains

  per-fold (section, domain) composition:
    section/domain    f  0 f  1 f  2
    sec00 source     330  330  330
    sec00 target       3    4    3
    sec01 source     330  330  330
    sec01 target       3    3    3
    sec02 source     330  330  330
    sec02 target       4    3    3
    sec03 source     330  330  330
    sec03 target       3    3    4
    sec04 source     330  330  330
    sec04 target       3    3    4
    sec05 source     330  330  330
    sec05 target       3    4    3
Will run folds: [0, 1, 2]

Full training pool

In [7]:
!python train_lean.py --mode domain_regularized --arch unet_mobilenet_encoder --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [8]:
!python train_lean.py --mode contrastive --arch unet_mobilenet_encoder --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [9]:
!python train_lean.py --mode mixed       --arch unet --epochs 15 --n_folds 5

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [10]:
!python train_lean.py --mode domain_regularized --arch unet --epochs 15 --n_folds 5

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [11]:
!python train_lean.py --mode contrastive --arch unet --epochs 15 --n_folds 5

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


## Step 2c: Train every mode with the **MobileNetV2** backbone

`MobileNetAE` uses an ImageNet-pretrained MobileNetV2 encoder (features[0:14], 96-channel @ 1/16 spatial) with a 4× transposed-conv decoder. ~2M parameters — biggest of the three backbones, but with cheap depthwise-separable convs it still trains in a reasonable time per fold. Log-mel is repeated to 3 channels so the pretrained ImageNet weights apply unchanged.

Same four DG modes as above. Each emits `checkpoints_phase2/{mode}_mobilenet_fold{0..4}.pt`. Step 4 will pick these up automatically alongside the `ae` and `unet` runs, and Step 5 will pick the winner across **all three architectures × all four modes** in one ranked leaderboard.

In [15]:
!python train_lean.py --mode mixed       --arch mobilenet --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [16]:
!python train_lean.py --mode domain_regularized --arch mobilenet --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


In [17]:
!python train_lean.py --mode contrastive --arch mobilenet --epochs 15 --n_folds 3

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
python3: can't open file 'train_phase2.py': [Errno 107] Transport endpoint is not connected


## Step 4: Evaluate every phase-2 (mode, arch) — K-fold ensemble on dev-test

For each `(mode, arch)` we discover every `{mode}_{arch}_fold{k}.pt` checkpoint under `checkpoints_phase2/`, score each fold on the **labelled dev-test** (sections 00-02), and then average the per-clip anomaly scores across folds to form the **ensemble**. We report:

- Per-fold source / target / overall AUC + pAUC + DCASE hmean (transparency, variance)
- **Ensemble** source / target / overall AUC + pAUC + DCASE hmean (primary number, comparable to phase 1)

**Eval-test (sections 03-05, unlabelled) is NOT scored here.** It is the held-out split we have reserved for a single final submission after the best `(mode, arch)` is picked from this leaderboard.

One failing evaluation no longer kills the loop (subprocess + clean stderr capture).

In [ ]:
import os, glob, json, re, subprocess, sys
from IPython.display import Image, Markdown, display

VALID_MODES   = {"baseline", "mixed", "domain_regularized", "contrastive"}
VALID_ARCHES  = {"ae", "unet", "mobilenet",}
CKPT_DIR      = os.environ.get("BEARING_PHASE2_CKPT_DIR", "/content/checkpoints_phase2")
EVAL_OUT_DIR  = "/content/eval_results_phase2"

# Group every {mode}_{arch}_fold{k}.pt checkpoint by (mode, arch).
FOLD_PATTERN = re.compile(r"^(?P<mode>[a-z]+)_(?P<arch>[a-z]+)_fold(?P<k>\d+)\.pt$")

seen_pairs = {}
for p in sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt"))):
    m = FOLD_PATTERN.match(os.path.basename(p))
    if not m: continue
    mode, arch = m.group("mode"), m.group("arch")
    if mode not in VALID_MODES or arch not in VALID_ARCHES: continue
    seen_pairs.setdefault((mode, arch), []).append(p)

if not seen_pairs:
    print(f"No fold checkpoints found under {CKPT_DIR} — run the training cells first.")

for (mode, arch), fold_paths in sorted(seen_pairs.items()):
    display(Markdown(f"### ▸ phase2 / {mode} · arch={arch}  "
                     f"({len(fold_paths)} fold checkpoints)"))
    cmd = [sys.executable, "evaluate_lean.py",
           "--mode", mode, "--arch", arch,
           "--out_dir", EVAL_OUT_DIR]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(f"[evaluate_lean.py failed for {mode}_{arch}]\n{proc.stderr}")
        continue

    run_dir = os.path.join(EVAL_OUT_DIR, f"{mode}_{arch}")
    summary_path = os.path.join(run_dir, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f: s = json.load(f)
        ens = s.get("ensemble", {})
        pdo = ens.get("per_domain_overall", {})
        ovr = ens.get("overall", {})
        src, tgt = pdo.get("source", {}), pdo.get("target", {})
        print(f"[ENSEMBLE] source  AUC={src.get('auc', float('nan')):.3f}  "
              f"pAUC={src.get('pauc', float('nan')):.3f}")
        print(f"[ENSEMBLE] target  AUC={tgt.get('auc', float('nan')):.3f}  "
              f"pAUC={tgt.get('pauc', float('nan')):.3f}")
        print(f"[ENSEMBLE] overall AUC={ovr.get('auc_pooled', float('nan')):.3f}  "
              f"pAUC={ovr.get('pauc_pooled', float('nan')):.3f}")
        print(f"[ENSEMBLE] DCASE harmonic mean = "
              f"{ens.get('harmonic_mean', float('nan')):.3f}")
        for f in s.get("per_fold", []):
            print(f"  fold {f['fold']}  src={f['source_auc']:.3f}  "
                  f"tgt={f['target_auc']:.3f}  ovr={f['overall_auc']:.3f}  "
                  f"hmean={f['harmonic_mean']:.3f}")

    # Display ensemble plots + the fold-vs-ensemble comparison.
    for fig in ["ensemble/overall_summary.png",
                "ensemble/score_histograms.png",
                "ensemble/roc.png",
                "ensemble/per_section_auc.png",
                "fold_vs_ensemble.png"]:
        path = os.path.join(run_dir, fig)
        if os.path.exists(path):
            display(Image(path))

## Step 5: Display all modes' results and pick the winner

The cell below:

1. Runs `collect_results_lean.py` to regenerate the leaderboard, plots, and `RESULTS_phase2.md` on local disk.
2. Parses `comparison.csv` and prints a **one-row-per-mode summary** of the ensemble metrics (source/target/overall AUC, DCASE hmean, and per-fold target-AUC σ).
3. Displays the ranked leaderboard bar chart, the source-vs-target scatter, and the K-fold variance plot inline.
4. Prints the **recommended model** to ship — picked by ensemble DCASE harmonic mean, with ties broken by target AUC and then by lowest per-fold variance.

In [ ]:
import csv, json, os, subprocess, sys
from IPython.display import Image, Markdown, display

RESULTS_LOCAL = "/content/results_phase2"
RESULTS_ZIP   = "/content/results_phase2.zip"

# 1) Regenerate leaderboard + plots + RESULTS_phase2.md + best_model/ on local disk.
proc = subprocess.run(
    [sys.executable, "collect_results_lean.py",
     "--ckpt_dir", os.environ["BEARING_PHASE2_CKPT_DIR"],
     "--eval_dir", EVAL_OUT_DIR,
     "--out_dir",  RESULTS_LOCAL,
     "--zip_name", RESULTS_ZIP,
     "--rank_by",  "hmean"],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print("collect_results_lean.py failed:\n", proc.stderr)

# 2) Per-mode table from comparison.csv.
csv_path = os.path.join(RESULTS_LOCAL, "comparison.csv")
rows = []
if os.path.exists(csv_path):
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))

if not rows:
    display(Markdown("_No runs found — make sure Step 4 finished for every mode._"))
else:
    header = (
        "| mode | arch | K | src AUC | tgt AUC | ovr AUC | DCASE hmean | "
        "fold tgt σ | src−tgt gap | mean val loss |"
    )
    sep = "| " + " | ".join(["---"] * 10) + " |"
    lines = [header, sep]
    rows.sort(key=lambda r: -float(r["hmean"] or "nan"))
    def f(v, n=4):
        try:    return f"{float(v):.{n}f}"
        except: return "—"
    for r in rows:
        lines.append(
            f"| {r['mode']} | {r['arch']} | {r['n_folds']} | "
            f"{f(r['src_auc'])} | {f(r['tgt_auc'])} | {f(r['ovr_auc'])} | "
            f"{f(r['hmean'])} | {f(r['fold_tgt_auc_std'])} | "
            f"{f(r['src_tgt_gap'])} | {f(r['mean_best_val_loss'])} |"
        )
    display(Markdown("### Per-mode ensemble summary (dev-test)\n" + "\n".join(lines)))

# 3) Inline plots.
for fig in ["leaderboard.png", "source_vs_target.png", "kfold_variance.png"]:
    p = os.path.join(RESULTS_LOCAL, fig)
    if os.path.exists(p):
        display(Image(p))

# 4) Recommended model.
winner_path = os.path.join(RESULTS_LOCAL, "best_model", "WINNER.json")
if os.path.exists(winner_path):
    with open(winner_path) as f:
        win = json.load(f)
    m = win["metrics"]
    display(Markdown(
        f"### ★ Recommended model: `{win['mode']}/{win['arch']}`\n"
        f"- **ensemble source AUC** = {m['src_auc']:.4f}\n"
        f"- **ensemble target AUC** = {m['tgt_auc']:.4f}\n"
        f"- **ensemble overall AUC** = {m['ovr_auc']:.4f}\n"
        f"- **ensemble DCASE hmean** = {m['hmean']:.4f}\n"
        f"- fold-to-fold target AUC σ = {m['fold_tgt_auc_std']:.4f}\n"
        f"- K-fold mean best val loss = {m['mean_best_val_loss']:.4f}\n\n"
        f"The K fold-checkpoints of this model are in `results_phase2/best_model/` "
        f"(inside the final zip bundle), ready to be handed to the eval-test "
        f"submission script once it is added."
    ))
else:
    display(Markdown("_No recommendation file produced — check stdout above for errors._"))

## Step 6: Sync bundle to Drive and download the zip

The bundle at `/content/results_phase2/` (and its zipped copy `/content/results_phase2.zip`) was already built in Step 5. This step:

1. Copies the whole `results_phase2/` tree to Drive at `<project>/results_phase2/` so it survives the runtime.
2. Copies the zip to Drive at `<project>/results_phase2.zip`.
3. Triggers a local download of the zip (for offline presentation / hand-off).

The zip contains:
- `RESULTS_phase2.md` — ranked leaderboard + per-fold variance + **recommended model** + repro steps
- `leaderboard.png`, `source_vs_target.png`, `kfold_variance.png`
- `checkpoints/` — every run's fold checkpoints + per-fold training histories
- `eval_results/` — full per-run evaluation artefacts (ensemble + per-fold plots + CSVs)
- `best_model/` — **K fold-checkpoints of the recommended model** + `WINNER.json`

In [ ]:
import os, shutil

PROJECT_ROOT = os.getcwd()  # set at Step 0 to the Drive project folder
DRIVE_BUNDLE = os.path.join(PROJECT_ROOT, "results_phase2")
DRIVE_ZIP    = os.path.join(PROJECT_ROOT, "results_phase2.zip")

# Mirror the bundle tree to Drive (wipes any previous copy).
if os.path.exists(DRIVE_BUNDLE):
    shutil.rmtree(DRIVE_BUNDLE)
shutil.copytree(RESULTS_LOCAL, DRIVE_BUNDLE)
print(f"Synced bundle to Drive   -> {DRIVE_BUNDLE}")

# Mirror the zip to Drive.
if os.path.exists(DRIVE_ZIP):
    os.remove(DRIVE_ZIP)
shutil.copy2(RESULTS_ZIP, DRIVE_ZIP)
print(f"Synced zip to Drive      -> {DRIVE_ZIP}")

print("\nLocal paths (cache):")
print(f"  {RESULTS_LOCAL}")
print(f"  {RESULTS_ZIP}")

In [ ]:
from IPython.display import Markdown, Image, display

report_path = os.path.join(RESULTS_LOCAL, "RESULTS_phase2.md")
if os.path.exists(report_path):
    with open(report_path) as f:
        display(Markdown(f.read()))

try:
    from google.colab import files
    files.download(RESULTS_ZIP)
    print(f"Download triggered for {RESULTS_ZIP}")
except Exception as e:
    print("Download skipped (not running in Colab):", e)

---

## Section B — Phase 3 (lean)

Using existing Colab session from Phase 2 (no remount).


In [ ]:

import os, json, glob, zipfile, shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
)
from IPython.display import Markdown, display

from model import build_model
from dataset import BearingDataset
from dataset_phase2 import BearingTestDataset

In [ ]:
# ---- paths ----
RESULTS_ZIP = "results_phase2.zip"
EXTRACT_DIR = "/content/results_phase2"
OUT_DIR = "/content/phase3_outputs"
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(RESULTS_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

winner_path = os.path.join(EXTRACT_DIR, "best_model", "WINNER.json")
with open(winner_path) as f:
    winner = json.load(f)

key = winner["key"]
arch = winner["arch"]
ckpts = sorted(glob.glob(os.path.join(EXTRACT_DIR, "best_model", f"{key}_fold*.pt")))
print("Winner:", winner)
print("Checkpoints:", len(ckpts))

In [ ]:
# ---- shared scoring helper ----
device = "cuda" if torch.cuda.is_available() else "cpu"

def score_with_ensemble(loader, arch, ckpt_paths):
    fold_scores = []
    domains_ref, sections_ref, labels_ref, files_ref = None, None, None, None

    for ckpt_path in ckpt_paths:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = build_model(arch).to(device)
        model.load_state_dict(ckpt["model_state"])
        model.eval()

        scores, domains, sections, labels, files = [], [], [], [], []
        with torch.no_grad():
            for batch in loader:
                x = batch[0].to(device, non_blocking=True)
                recon, _ = model(x)
                t = min(recon.shape[-1], x.shape[-1])
                err = ((recon[..., :t] - x[..., :t]) ** 2).mean(dim=[1, 2, 3]).cpu().numpy()
                scores.append(err)

                # For BearingDataset(return_label=True): (x, domain, section, label)
                # For BearingTestDataset:               (x, domain, section)
                dom = list(batch[1])
                sec = [int(s) for s in batch[2]]
                lab = list(batch[3]) if len(batch) > 3 else ["unknown"] * len(dom)
                domains.extend(dom)
                sections.extend(sec)
                labels.extend(lab)

        fold_scores.append(np.concatenate(scores))
        if domains_ref is None:
            domains_ref = np.array(domains)
            sections_ref = np.array(sections)
            labels_ref = np.array(labels)

    ens_scores = np.mean(np.stack(fold_scores, axis=0), axis=0)
    return ens_scores, domains_ref, sections_ref, labels_ref

In [ ]:
# ---- 1) Calibrate threshold on labelled dev_test (domain-agnostic) ----
dev_test_dir = "data/dcase_bearing_dev/bearing/test"
dev_test_cache = "/content/bearing_cache_dev_test"
dev_ds = BearingDataset(dev_test_dir, cache_dir=dev_test_cache, return_label=True, verbose=False)
dev_loader = DataLoader(dev_ds, batch_size=128, shuffle=False, num_workers=0)

dev_scores, dev_domains, dev_sections, dev_labels = score_with_ensemble(dev_loader, arch, ckpts)
y_true = (dev_labels == "anomaly").astype(int)

prec, rec, thr = precision_recall_curve(y_true, dev_scores)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
best_i = int(np.nanargmax(f1))
best_thr = float(thr[best_i])
y_pred = (dev_scores >= best_thr).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
metrics = {
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "roc_auc": float(roc_auc_score(y_true, dev_scores)),
    "average_precision": float(average_precision_score(y_true, dev_scores)),
    "threshold": best_thr,
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
}

display(Markdown(
    f"""
### Domain-agnostic threshold (calibrated on dev_test)
- threshold = `{metrics['threshold']:.6f}`
- precision = `{metrics['precision']:.4f}`
- recall = `{metrics['recall']:.4f}`
- F1 = `{metrics['f1']:.4f}`
- accuracy = `{metrics['accuracy']:.4f}`
- ROC-AUC = `{metrics['roc_auc']:.4f}`
- PR-AUC = `{metrics['average_precision']:.4f}`
- confusion = `TN={tn}, FP={fp}, FN={fn}, TP={tp}`
"""
))

with open(os.path.join(OUT_DIR, "threshold_metrics.json"), "w") as f:
    json.dump({"winner": winner, "domain_agnostic": True, "metrics": metrics}, f, indent=2)

## Explainability (what separates normal vs anomaly)

This section is **post-hoc** explanation of the winner's anomaly score = mean squared reconstruction error (MSE).

### What `domain_regularized` changed during training

`domain_regularized` adds a small domain-classification loss on the bottleneck features `z` (source vs target). At **inference**, we still score clips with reconstruction MSE only — the domain head is not used.

So explainability here answers: **which time-frequency regions drive high reconstruction error for anomalies** (and whether that pattern is consistent across sections/domains).

### Plots produced

- `explain_melbin_mean_error.png` — average squared error per mel bin for normals vs anomalies (pooled dev-test).
- `explain_pca_latent.png` — PCA of pooled bottleneck vectors `z` (2D), colored by label/section/domain.
- `explain_examples/` — a few side-by-side panels: input log-mel, reconstruction, squared error map.

In [ ]:
# ---- 1b) Explainability on dev_test (winner ensemble) ----
from sklearn.decomposition import PCA

explain_dir = os.path.join(OUT_DIR, "explain")
os.makedirs(explain_dir, exist_ok=True)

models = []
for ckpt_path in ckpts:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = build_model(arch).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    models.append(m)

# --- A) Mel-bin attribution: mean squared error averaged over time ---
mel_err_sum = {"normal": None, "anomaly": None}
mel_err_cnt = {"normal": 0, "anomaly": 0}

# --- B) Latent PCA features (pooled z) ---
Z_list, y_list, dom_list, sec_list = [], [], [], []

with torch.no_grad():
    for batch in dev_loader:
        x = batch[0].to(device, non_blocking=True)
        dom = list(batch[1])
        sec = [int(s) for s in batch[2]]
        lab = list(batch[3])

        # fold-mean squared error map + fold-mean recon for viz
        err_maps = []
        recons = []
        zs = []
        for m in models:
            recon, z = m(x)
            t = min(recon.shape[-1], x.shape[-1])
            e = (recon[..., :t] - x[..., :t]) ** 2  # (B,1,H,T)
            err_maps.append(e.detach())
            recons.append(recon[..., :t].detach())
            zp = z.mean(dim=[2, 3]).detach().cpu().numpy()
            zs.append(zp)

        err_map = torch.stack(err_maps, dim=0).mean(dim=0)  # (B,1,H,T)
        recon_m = torch.stack(recons, dim=0).mean(dim=0)
        z_ens = np.mean(np.stack(zs, axis=0), axis=0)  # (B, C)

        # mel-bin curves (H,)
        mel_curve = err_map.mean(dim=(1, 3)).squeeze(1).cpu().numpy()  # (B, H)

        for i in range(x.size(0)):
            tag = "anomaly" if lab[i] == "anomaly" else "normal"
            v = mel_curve[i]
            if mel_err_sum[tag] is None:
                mel_err_sum[tag] = v.astype(np.float64)
            else:
                mel_err_sum[tag] += v.astype(np.float64)
            mel_err_cnt[tag] += 1

            Z_list.append(z_ens[i])
            y_list.append(1 if lab[i] == "anomaly" else 0)
            dom_list.append(dom[i])
            sec_list.append(sec[i])

bins = np.arange(mel_err_sum["normal"].shape[0])
plt.figure(figsize=(9, 4.5))
plt.plot(bins, mel_err_sum["normal"] / max(mel_err_cnt["normal"], 1), label="normal (mean sq err / mel bin)")
plt.plot(bins, mel_err_sum["anomaly"] / max(mel_err_cnt["anomaly"], 1), label="anomaly (mean sq err / mel bin)")
plt.plot(
    bins,
    (mel_err_sum["anomaly"] / max(mel_err_cnt["anomaly"], 1))
    - (mel_err_sum["normal"] / max(mel_err_cnt["normal"], 1)),
    label="difference (anomaly - normal)",
    color="k",
    alpha=0.6,
)
plt.xlabel("mel bin")
plt.ylabel("mean squared reconstruction error")
plt.title("Mel-bin attribution (dev-test, domain pooled)")
plt.grid(alpha=0.3)
plt.legend()
p_mel = os.path.join(explain_dir, "explain_melbin_mean_error.png")
plt.tight_layout()
plt.savefig(p_mel, dpi=140)
plt.close()

Z = np.stack(Z_list, axis=0)
y = np.array(y_list)
pca = PCA(n_components=2, random_state=0)
Z2 = pca.fit_transform(Z)

plt.figure(figsize=(6.2, 5.2))
plt.scatter(Z2[y == 0, 0], Z2[y == 0, 1], s=18, alpha=0.55, label="normal")
plt.scatter(Z2[y == 1, 0], Z2[y == 1, 1], s=18, alpha=0.55, label="anomaly")
plt.title("PCA of pooled bottleneck z (ensemble mean across folds)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca = os.path.join(explain_dir, "explain_pca_latent_label.png")
plt.tight_layout()
plt.savefig(p_pca, dpi=140)
plt.close()

sec_arr = np.array(sec_list)
plt.figure(figsize=(6.2, 5.2))
for s in sorted(set(sec_arr.tolist())):
    m = sec_arr == s
    plt.scatter(Z2[m, 0], Z2[m, 1], s=18, alpha=0.55, label=f"sec {s}")
plt.title("PCA of z colored by section")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca_sec = os.path.join(explain_dir, "explain_pca_latent_section.png")
plt.tight_layout()
plt.savefig(p_pca_sec, dpi=140)
plt.close()

dom_arr = np.array(dom_list)
plt.figure(figsize=(6.2, 5.2))
for d in ["source", "target"]:
    m = dom_arr == d
    plt.scatter(Z2[m, 0], Z2[m, 1], s=18, alpha=0.55, label=d)
plt.title("PCA of z colored by domain")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca_dom = os.path.join(explain_dir, "explain_pca_latent_domain.png")
plt.tight_layout()
plt.savefig(p_pca_dom, dpi=140)
plt.close()

# --- C) Example panels: pick a few highest-error anomalies + a few lowest-error normals ---

def _example_panel(idx, tag):
    sample = dev_ds.samples[idx]
    x = dev_ds[idx][0].unsqueeze(0).to(device)
    with torch.no_grad():
        err_maps = []
        recons = []
        for m in models:
            recon, _ = m(x)
            t = min(recon.shape[-1], x.shape[-1])
            e = (recon[..., :t] - x[..., :t]) ** 2
            err_maps.append(e)
            recons.append(recon[..., :t])
        err = torch.stack(err_maps, dim=0).mean(dim=0).detach().cpu().numpy()
        err = np.squeeze(err)  # (H, T)
        rec = torch.stack(recons, dim=0).mean(dim=0).detach().cpu().numpy()
        rec = np.squeeze(rec)
    inp = x.detach().cpu().numpy()
    inp = np.squeeze(inp)
    if inp.ndim != 2:
        raise ValueError(f"expected 2D log-mel for imshow, got shape {inp.shape}")

    vmax = float(np.percentile(err, 99.0))
    fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
    ax[0].imshow(inp, aspect="auto", origin="lower")
    ax[0].set_title("input log-mel")
    ax[1].imshow(rec, aspect="auto", origin="lower")
    ax[1].set_title("recon (ens.)")
    im = ax[2].imshow(err, aspect="auto", origin="lower", vmin=0.0, vmax=max(vmax, 1e-8))
    ax[2].set_title("squared error")
    for a in ax:
        a.set_xlabel("time")
        a.set_ylabel("mel")
    fig.suptitle(f"{tag}: {sample['filename']}  (label={sample['label']}, dom={sample['domain']}, sec={sample['section']})")
    fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
    fig.tight_layout()
    out = os.path.join(explain_dir, f"example_{tag}_{idx}.png")
    fig.savefig(out, dpi=160)
    plt.close(fig)
    return out

order = np.argsort(-dev_scores)  # high error first
anom_idx = [i for i in order.tolist() if dev_ds.samples[i]["label"] == "anomaly"][:3]
norm_idx = [i for i in np.argsort(dev_scores).tolist() if dev_ds.samples[i]["label"] == "normal"][:3]

example_paths = []
for i in anom_idx:
    example_paths.append(_example_panel(i, "high_err_anomaly"))
for i in norm_idx:
    example_paths.append(_example_panel(i, "low_err_normal"))

summary_x = {
    "winner": winner,
    "note": "Explainability is based on reconstruction error and bottleneck embeddings; domain classifier is not used at inference.",
    "plots": {
        "melbin": p_mel,
        "pca_label": p_pca,
        "pca_section": p_pca_sec,
        "pca_domain": p_pca_dom,
        "examples": example_paths,
    },
}
with open(os.path.join(explain_dir, "explain_summary.json"), "w") as f:
    json.dump(summary_x, f, indent=2)

display(Markdown("### Explainability plots"))
for path in [p_mel, p_pca, p_pca_sec, p_pca_dom] + example_paths:
    display(Image(filename=path))


In [ ]:
# ---- 2) Score unlabeled eval_test and write per-section CSVs ----
eval_test_dir = "data/dcase_bearing_eval/bearing/test"
eval_test_cache = "/content/bearing_cache_eval_test"
eval_ds = BearingTestDataset(eval_test_dir, cache_dir=eval_test_cache, verbose=False)
eval_loader = DataLoader(eval_ds, batch_size=128, shuffle=False, num_workers=0)

eval_scores, eval_domains, eval_sections, eval_labels = score_with_ensemble(eval_loader, arch, ckpts)
eval_pred = (eval_scores >= best_thr).astype(int)

records = []
for i, s in enumerate(eval_ds.samples):
    records.append({
        "filename": s["filename"],
        "section": int(s["section"]),
        "score": float(eval_scores[i]),
        "decision": int(eval_pred[i]),
    })

sections = sorted(set(r["section"] for r in records))
for sec in sections:
    sec_rows = sorted([r for r in records if r["section"] == sec], key=lambda x: x["filename"])
    score_path = os.path.join(OUT_DIR, f"anomaly_score_section_{sec:02d}.csv")
    decision_path = os.path.join(OUT_DIR, f"decision_result_section_{sec:02d}.csv")
    with open(score_path, "w") as f:
        for r in sec_rows:
            f.write(f"{r['filename']},{r['score']:.10f}\n")
    with open(decision_path, "w") as f:
        for r in sec_rows:
            f.write(f"{r['filename']},{r['decision']}\n")

summary = {
    "winner": winner,
    "threshold": best_thr,
    "n_eval_clips": len(records),
    "sections": sections,
    "mean_score": float(np.mean(eval_scores)),
    "std_score": float(np.std(eval_scores)),
    "predicted_anomaly_rate": float(np.mean(eval_pred)),
}
with open(os.path.join(OUT_DIR, "eval_test_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

display(Markdown(
    f"""
### Eval-test scoring complete
- clips scored: `{len(records)}`
- sections: `{sections}`
- threshold used: `{best_thr:.6f}`
- predicted anomaly rate: `{np.mean(eval_pred):.4f}`
- outputs: `anomaly_score_section_XX.csv`, `decision_result_section_XX.csv`
"""
))

In [ ]:
# ---- 3) zip outputs + copy to Drive + download ----
zip_base = "/content/phase3_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Created:", zip_path)

drive_zip = os.path.join(os.getcwd(), "phase3_outputs.zip")
if os.path.exists(drive_zip):
    os.remove(drive_zip)
shutil.copy2(zip_path, drive_zip)
print("Copied to Drive project folder:", drive_zip)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download skipped:", e)

---

## Section C — Phase 4 (lean)

Using existing Colab session from Phase 2 (no remount).


In [ ]:

import os, json, glob, zipfile, shutil
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import Markdown, display, Image
from sklearn.decomposition import PCA

from model import build_model
from dataset import BearingDataset

In [ ]:
# ---- knobs ----
RESULTS_ZIP = "results_phase2.zip"
EXTRACT_DIR = "/content/results_phase2"
OUT_DIR = "/content/phase4_outputs"
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEV_TEST_DIR = "data/dcase_bearing_dev/bearing/test"
DEV_TEST_CACHE = "/content/bearing_cache_dev_test"

BATCH_SIZE = 32
N_EXAMPLES_HIGH = 6   # high-MSE anomalies
N_EXAMPLES_LOW = 6    # low-MSE normals
IG_STEPS = 32         # integrated-gradients steps (higher = smoother, slower)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# ---- load winner checkpoints ----
with zipfile.ZipFile(RESULTS_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

winner_path = os.path.join(EXTRACT_DIR, "best_model", "WINNER.json")
with open(winner_path) as f:
    winner = json.load(f)

key = winner["key"]
arch = winner["arch"]
ckpts = sorted(glob.glob(os.path.join(EXTRACT_DIR, "best_model", f"{key}_fold*.pt")))
if not ckpts:
    raise FileNotFoundError(f"No checkpoints for {key} under {EXTRACT_DIR}/best_model")

models = []
for p in ckpts:
    ckpt = torch.load(p, map_location=device, weights_only=False)
    m = build_model(arch).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    models.append(m)

display(Markdown(f"### Winner\n- `{winner['mode']}/{winner['arch']}`\n- folds: {len(ckpts)}\n- key: `{key}`"))

In [ ]:
# ---- dataset + ensemble MSE vector ----
dev_ds = BearingDataset(DEV_TEST_DIR, cache_dir=DEV_TEST_CACHE, return_label=True, verbose=False)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        pin_memory=(device == "cuda"))

def ensemble_forward(x):
    """Return ensemble-mean recon and per-fold recons. x: (B,1,H,T)"""
    recons = []
    for m in models:
        recon, _ = m(x)
        recons.append(recon)
    # align time
    t = min(r.shape[-1] for r in recons + [x])
    x0 = x[..., :t]
    recons_t = [r[..., :t] for r in recons]
    recon_mean = torch.stack(recons_t, dim=0).mean(dim=0)
    return recon_mean, recons_t, x0

@torch.no_grad()
def per_clip_mse(x):
    recon_mean, _, x0 = ensemble_forward(x)
    return ((recon_mean - x0) ** 2).mean(dim=(1, 2, 3))

all_scores = []
all_dom, all_sec, all_lab, all_name = [], [], [], []
with torch.no_grad():
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        s = per_clip_mse(x).detach().cpu().numpy()
        all_scores.append(s)
        all_dom.extend(list(dom))
        all_sec.extend([int(v) for v in sec])
        all_lab.extend(list(lab))

scores = np.concatenate(all_scores)
domains = np.array(all_dom)
sections = np.array(all_sec)
labels = np.array(all_lab)
names = np.array([dev_ds.samples[i]["filename"] for i in range(len(dev_ds))])

display(Markdown(f"### dev_test scored\n- clips: {len(scores)}"))


In [ ]:
# ---- A) mel-bin mean squared error curves (label × section × domain pooled variants) ----
explain_dir = os.path.join(OUT_DIR, "explain")
os.makedirs(explain_dir, exist_ok=True)

mel_sum = {}
mel_cnt = {}

def _key(lab, sec=None, dom=None):
    if sec is None and dom is None:
        return f"label={lab}"
    if sec is None:
        return f"label={lab}|dom={dom}"
    if dom is None:
        return f"label={lab}|sec={sec}"
    return f"label={lab}|sec={sec}|dom={dom}"

with torch.no_grad():
    idx = 0
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        recon_mean, _, x0 = ensemble_forward(x)
        e = (recon_mean - x0) ** 2  # (B,1,H,T)
        mel = e.mean(dim=(1, 3)).squeeze(1).detach().cpu().numpy()  # (B,H)
        for i in range(x.size(0)):
            l = lab[i]
            s = int(sec[i])
            d = dom[i]
            for k in [
                _key(l),
                _key(l, sec=s),
                _key(l, dom=d),
                _key(l, sec=s, dom=d),
            ]:
                mel_sum[k] = mel_sum.get(k, None)
                if mel_sum[k] is None:
                    mel_sum[k] = mel[i].astype(np.float64)
                else:
                    mel_sum[k] += mel[i]
                mel_cnt[k] = mel_cnt.get(k, 0) + 1
        idx += x.size(0)

# pooled normal vs anomaly
def _curve(k):
    return mel_sum[k] / max(mel_cnt.get(k, 0), 1)

k_n, k_a = _key("normal"), _key("anomaly")
bins = np.arange(_curve(k_n).shape[0])
plt.figure(figsize=(9, 4.6))
plt.plot(bins, _curve(k_n), label="normal")
plt.plot(bins, _curve(k_a), label="anomaly")
plt.plot(bins, _curve(k_a) - _curve(k_n), color="k", alpha=0.6, label="anomaly - normal")
plt.title("Mel-bin mean squared reconstruction error (domain+section pooled)")
plt.xlabel("mel bin")
plt.ylabel("mean squared error")
plt.grid(alpha=0.3)
plt.legend()
p0 = os.path.join(explain_dir, "melbin_pooled_label.png")
plt.tight_layout(); plt.savefig(p0, dpi=140); plt.close()

# per section (00-02)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, sec in zip(axes, [0, 1, 2]):
    kn, ka = _key("normal", sec=sec), _key("anomaly", sec=sec)
    ax.plot(bins, _curve(kn), label="normal")
    ax.plot(bins, _curve(ka), label="anomaly")
    ax.plot(bins, _curve(ka) - _curve(kn), color="k", alpha=0.45, label="diff")
    ax.set_title(f"section {sec:02d}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("mean squared error")
axes[-1].legend(loc="upper right")
p1 = os.path.join(explain_dir, "melbin_by_section.png")
plt.tight_layout(); plt.savefig(p1, dpi=140); plt.close()

display(Markdown("### Mel-bin attribution"))
display(Image(p0)); display(Image(p1))

In [ ]:
# ---- B) PCA of pooled bottleneck z (ensemble mean) ----
Z, y, dlist, slist = [], [], [], []
with torch.no_grad():
    for x, dom, sec, lab in dev_loader:
        x = x.to(device, non_blocking=True)
        zs = []
        for m in models:
            _, z = m(x)
            zp = z.mean(dim=[2, 3]).detach().cpu().numpy()
            zs.append(zp)
        z_ens = np.mean(np.stack(zs, axis=0), axis=0)
        for i in range(x.size(0)):
            Z.append(z_ens[i])
            y.append(1 if lab[i] == "anomaly" else 0)
            dlist.append(dom[i])
            slist.append(int(sec[i]))

Z = np.stack(Z, axis=0)
y = np.array(y)
darr = np.array(dlist)
sarr = np.array(slist)

Z2 = PCA(n_components=2, random_state=0).fit_transform(Z)

def _scatter(mask, title, path, legend=True):
    plt.figure(figsize=(6.2, 5.2))
    plt.scatter(Z2[mask, 0], Z2[mask, 1], s=16, alpha=0.55)
    plt.title(title)
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.grid(alpha=0.3)
    if legend:
        plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
plt.scatter(Z2[y == 0, 0], Z2[y == 0, 1], s=16, alpha=0.55, label="normal")
plt.scatter(Z2[y == 1, 0], Z2[y == 1, 1], s=16, alpha=0.55, label="anomaly")
plt.title("PCA of pooled z (colored by label)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_lab = os.path.join(explain_dir, "pca_z_label.png")
plt.tight_layout(); plt.savefig(p_pca_lab, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
for d in ["source", "target"]:
    m = darr == d
    plt.scatter(Z2[m, 0], Z2[m, 1], s=16, alpha=0.55, label=d)
plt.title("PCA of pooled z (colored by domain)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_dom = os.path.join(explain_dir, "pca_z_domain.png")
plt.tight_layout(); plt.savefig(p_pca_dom, dpi=140); plt.close()

plt.figure(figsize=(6.2, 5.2))
for s in sorted(set(sarr.tolist())):
    m = sarr == s
    plt.scatter(Z2[m, 0], Z2[m, 1], s=16, alpha=0.55, label=f"sec {s}")
plt.title("PCA of pooled z (colored by section)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(alpha=0.3); plt.legend()
p_pca_sec = os.path.join(explain_dir, "pca_z_section.png")
plt.tight_layout(); plt.savefig(p_pca_sec, dpi=140); plt.close()

display(Markdown("### Latent PCA"))
display(Image(p_pca_lab)); display(Image(p_pca_dom)); display(Image(p_pca_sec))

In [ ]:
# ---- C) Integrated Gradients of scalar MSE w.r.t. input log-mel ----
def integrated_gradients_mse(x0, steps=32):
    """Integrated gradients of scalar MSE(recon, x) w.r.t. input x.

    x0 must be detached from any previous graph (we clone internally).
    """
    x_ref = x0.detach().clone()
    baseline = torch.zeros_like(x_ref)
    grads = []
    for k in range(1, steps + 1):
        alpha = float(k) / float(steps)
        x = baseline + alpha * (x_ref - baseline)
        x = x.detach().requires_grad_(True)

        recon_mean, _, xt = ensemble_forward(x)
        t = min(recon_mean.shape[-1], xt.shape[-1])
        mse = ((recon_mean[..., :t] - xt[..., :t]) ** 2).mean()
        mse.backward()
        grads.append(x.grad.detach())

        for m in models:
            m.zero_grad(set_to_none=True)

    avg_grad = torch.stack(grads, dim=0).mean(dim=0)
    ig = (x_ref - baseline) * avg_grad
    return ig.squeeze(0).squeeze(0).detach().cpu().numpy()  # (H,T)

def save_triplet(inp, recon, err, ig, title, path):
    inp = np.squeeze(inp); recon = np.squeeze(recon); err = np.squeeze(err); ig = np.squeeze(ig)
    fig, ax = plt.subplots(1, 4, figsize=(16.5, 3.4))
    ax[0].imshow(inp, aspect="auto", origin="lower"); ax[0].set_title("input")
    ax[1].imshow(recon, aspect="auto", origin="lower"); ax[1].set_title("recon (ens.)")
    vmax = float(np.percentile(err, 99.0))
    im2 = ax[2].imshow(err, aspect="auto", origin="lower", vmin=0.0, vmax=max(vmax, 1e-8))
    ax[2].set_title("squared error")
    vmax3 = float(np.percentile(np.abs(ig), 99.0))
    im3 = ax[3].imshow(ig, aspect="auto", origin="lower", cmap="coolwarm",
                     vmin=-vmax3, vmax=vmax3)
    ax[3].set_title("IG(MSE) wrt input")
    for a in ax:
        a.set_xlabel("time"); a.set_ylabel("mel")
    fig.suptitle(title)
    fig.colorbar(im2, ax=ax[2], fraction=0.046, pad=0.04)
    fig.colorbar(im3, ax=ax[3], fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(path, dpi=170); plt.close(fig)

order = np.argsort(-scores)
anom_idx = [i for i in order.tolist() if labels[i] == "anomaly"][:N_EXAMPLES_HIGH]
norm_idx = [i for i in np.argsort(scores).tolist() if labels[i] == "normal"][:N_EXAMPLES_LOW]

ig_paths = []
for i in anom_idx + norm_idx:
    tag = "anom" if labels[i] == "anomaly" else "norm"
    x = dev_ds[i][0].unsqueeze(0).to(device)
    x0 = x.detach()

    recon_mean, _, x0a = ensemble_forward(x0)
    t = min(recon_mean.shape[-1], x0a.shape[-1])
    inp = x0a[..., :t].squeeze(0).squeeze(0).detach().cpu().numpy()
    rec = recon_mean[..., :t].squeeze(0).squeeze(0).detach().cpu().numpy()
    err = (recon_mean[..., :t] - x0a[..., :t]).pow(2).squeeze(0).squeeze(0).detach().cpu().numpy()

    ig = integrated_gradients_mse(x0, steps=IG_STEPS)
    ig = ig[:, :t]

    title = f"{tag} | {names[i]} | score={float(scores[i]):.6f}"
    outp = os.path.join(explain_dir, f"ig_{tag}_{i}.png")
    save_triplet(inp, rec, err, ig, title, outp)
    ig_paths.append(outp)

display(Markdown("### Integrated gradients examples"))
for pth in ig_paths:
    display(Image(pth))


In [ ]:
# ---- D) bundle outputs ----
summary = {
    "winner": winner,
    "outputs": {
        "explain_dir": explain_dir,
        "melbin_pooled": p0,
        "melbin_by_section": p1,
        "pca_label": p_pca_lab,
        "pca_domain": p_pca_dom,
        "pca_section": p_pca_sec,
        "integrated_gradients": ig_paths,
    },
}
with open(os.path.join(OUT_DIR, "phase4_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive("/content/phase4_outputs", "zip", OUT_DIR)
print("zip:", zip_path)

drive_zip = os.path.join(os.getcwd(), "phase4_outputs.zip")
if os.path.exists(drive_zip):
    os.remove(drive_zip)
shutil.copy2(zip_path, drive_zip)
print("copied to:", drive_zip)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download skipped:", e)